# Data laoding benchmarke from dCache and project space

Experiment reading huge zarr file from dCache and project space. This experiment:

- runs on a jupyterdask session with 24 cores and 187GB mem on interactive node (with small and lag)
- load the `real` data variable from the zarr storage, which is ~87GB according to Dask graph
- since we are only loading a data variable, the dask graph gets slightly longer because of the data variable selection. In the end, all coodinates and one data variable are loaded
- runs within a Python env containerized as Apptainer
- uses magic command `%%timeit`, with default setup 7 iterations and 1 loop per iteration
- restarts kernel between each benchmark to avoid potential caching

In [1]:
# import depsi
import xarray as xr
from pathlib import Path
import json

import fsspec
from dcachefs import dCacheFileSystem

## Configure paths

In [2]:
# Huge data ~228GB
path_huge_dcache = 'dcache://disk/Data/ou_dcache_benchmark/nl_groningen_s1_asc_t088.zarr'
path_huge_pjspace = Path('/project/caroline/Share/stacks_zarr/nl_groningen_s1_asc_t088/nl_groningen_s1_asc_t088.zarr')

## Inspect the chunk sizes, also validate that files are consolidated

In [3]:
# Huge dchacge
zarr_store = fsspec.get_mapper(
    path_huge_dcache,
    api_url='https://dcacheview.grid.surfsara.nl:22880/api/v1',
    webdav_url='https://webdav.grid.surfsara.nl:2880/',
)
stack_huge_dcache = xr.open_zarr(zarr_store, consolidated=True)

# Huge pjspace
stack_huge_pjspace = xr.open_zarr(path_huge_pjspace, consolidated=True)

In [4]:
print(f"huge data dcache, nr. of chunks: azimuth: {len(stack_huge_dcache.chunks['azimuth'])}, range {len(stack_huge_dcache.chunks['range'])}, time {len(stack_huge_dcache.chunks['time'])}")
print(f"huge data pjspace, nr. of chunks: azimuth: {len(stack_huge_pjspace.chunks['azimuth'])}, range {len(stack_huge_pjspace.chunks['range'])}, time {len(stack_huge_pjspace.chunks['time'])}")

huge data dcache, nr. of chunks: azimuth: 2, range 5, time 511
huge data pjspace, nr. of chunks: azimuth: 2, range 5, time 511


## Exp1: Huge file from dcahce

In [ ]:
%%timeit
# Input dcache store 
zarr_store = fsspec.get_mapper(
    path_huge_dcache,
    api_url='https://dcacheview.grid.surfsara.nl:22880/api/v1',
    webdav_url='https://webdav.grid.surfsara.nl:2880/',
)

# Lazy loading stack
stack_huge_dcache = xr.open_zarr(zarr_store)
# stack_huge_dcache = stack_huge_dcache.compute()
stack_huge_dcache_real = stack_huge_dcache['real'].compute() # Only load "real" field, ~87GBs

9min 48s ± 5.19 s per loop (mean ± std. dev. of 7 runs, 1 loop each)


## Exp2: Huge file from project space

In [ ]:
%%timeit
stack_huge_pjspace = xr.open_zarr(path_huge_pjspace)
# stack_huge_pjspace = stack_huge_pjspace.compute()
stack_huge_pjspace_real = stack_huge_pjspace['real'].compute()  # Only load "real" field, ~87GBs

9min 52s ± 1min 56s per loop (mean ± std. dev. of 7 runs, 1 loop each)


## Results
- huge data dcache: 9min 48s ± 5.19 s per loop (mean ± std. dev. of 7 runs, 1 loop each)
- huge data project space: 9min 52s ± 1min 56s per loop (mean ± std. dev. of 7 runs, 1 loop each)